In [ ]:
import pandas as pd 
import os
import numpy as np 
from pathlib import Path
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import ast
import sys 
import shutil
import json

# Custom imports from utils.py
script_dir = os.path.dirname(os.path.abspath(__file__))
repo_path = script_dir.split("WP2_Sweet_Protein_Discovery")[0] + "/WP2_Sweet_Protein_Discovery"
sys.path.append(repo_path)
from utils import get_best_of3_model, calculate_tm_score, avg_plddt, get_contacts

repo_path = os.path.dirname(os.getcwd())
repo_path = repo_path.split("WP2_Sweet_Protein_Discovery")[0] + "WP2_Sweet_Protein_Discovery"
monomer_path = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/monomers/"
complex_path = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/complexes/"

# Read dataframe of representative hits 
representative_structures_df = pd.read_csv(f"{repo_path}/6_structural_analyses/3_representative_structures/representative_structures.csv")
representative_structures_df 

,query,target,evalue,bits,pident,fident,nident,qstart,qend,qlen,...,af3_avg_pLDDT,of3_avg_pLDDT,model,source,length_diff,interface_residues,grantham_score,pdb,cropped_tseq,cluster_id
0,MNEI_1IV7,UniProt_tr|A0A0D9WYJ2|A0A0D9WYJ2_9ORYZ,3.760000e-04,273,32.9,0.329,32,1,90,96,...,NaN,NaN,AFDB,UniProt,0.052083,"[('E', 22, 'E'), ('Q', 27, 'Q'), ('C', 40, 'G'...",64.142857,MNEI_1IV7___A0A0D9WYJ2,DAWSPIKNINDPYIQKLGWWAVSEENKVSQSGALTFSKVTGGEQQL...,NaN
1,MNEI_1IV7,UniProt_tr|A0A0D3FFA6|A0A0D3FFA6_9ORYZ,7.672000e-04,260,30.9,0.309,30,1,90,96,...,NaN,NaN,AFDB,UniProt,0.062500,"[('E', 22, 'E'), ('Q', 27, 'A'), ('C', 40, 'G'...",64.571429,MNEI_1IV7___A0A0D3FFA6,GSWEPVDINDPHVQELGRWAVAEEDRGVAAGGLTFERVTDGEKQVV...,NaN
2,MNEI_1IV7,UniProt_tr|A0A0D9Z3S2|A0A0D9Z3S2_9ORYZ,7.672000e-04,260,30.9,0.309,30,1,90,96,...,NaN,NaN,AFDB,UniProt,0.062500,"[('E', 22, 'E'), ('Q', 27, 'A'), ('C', 40, 'G'...",64.571429,MNEI_1IV7___A0A0D9Z3S2,GSWEPVDINDPHVQELGRWAVAEEDRGVAAGGLTFERVTDGEKQVV...,NaN
3,MNEI_1IV7,UniProt_tr|A0A368SP17|A0A368SP17_SETIT,9.555000e-04,256,25.5,0.255,25,1,90,96,...,NaN,NaN,AFDB,UniProt,0.020833,"[('E', 22, 'A'), ('Q', 27, 'L'), ('C', 40, 'G'...",114.428571,MNEI_1IV7___A0A368SP17,GGWSAIQNVSDPHIQELGGWALGQAKEKQLAGAGLRFRRVVRGEQQ...,NaN
4,MNEI_1IV7,UniProt_tr|A0A4U6T2U9|A0A4U6T2U9_SETVI,9.555000e-04,256,25.5,0.255,25,1,90,96,...,NaN,NaN,AFDB,UniProt,0.031250,"[('E', 22, 'A'), ('Q', 27, 'L'), ('C', 40, 'G'...",114.428571,MNEI_1IV7___A0A4U6T2U9,GGWSAIQNVSDPHIQELGGWALGQAKEKQLAGAGLRFRRVVRGEQQ...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3671,Thaumatin-I_1LR2,UniProt_tr|W9R8D6|W9R8D6_9ROSA,1.839000e-19,891,40.3,0.403,90,1,205,207,...,NaN,NaN,AFDB,UniProt,0.072464,"[('K', 66, 'I'), ('K', 77, 'G'), ('K', 96, 'Q'...",63.375000,Thaumatin-I_1LR2___W9R8D6,AKVTIKSNCPRTIWPATLTGDNKAQLSTTGFKLSKGQSRSVVLPAP...,Thaumatin-I_1LR2___W9R8D6
3672,Thaumatin-I_1LR2,UniProt_tr|W9RV70|W9RV70_9ROSA,5.628000e-20,912,40.7,0.407,88,2,204,207,...,NaN,NaN,AFDB,UniProt,0.038647,"[('K', 66, 'E'), ('K', 77, 'T'), ('K', 96, 'I'...",64.625000,Thaumatin-I_1LR2___W9RV70,RVFTVVNYCKEAIWPAITNAENITGGGFALKPGQSAVYTAPLGWHG...,Thaumatin-I_1LR2___W9RV70
3673,Thaumatin-II_3AOK,UniProt_tr|W9S414|W9S414_9ROSA,2.988000e-20,912,39.3,0.393,89,2,206,207,...,NaN,NaN,AFDB,UniProt,0.096618,"[('R', 66, 'V'), ('K', 77, 'G'), ('R', 81, 'T'...",87.222222,Thaumatin-II_3AOK___W9S414,TTFTLLNKCEYTVWPGILSNAGVSPLPTTGFVLQKGESKTLTAPSG...,Thaumatin-I_1LR2___W9S414
3674,Thaumatin-I_1LR2,UniProt_tr|W9SSV3|W9SSV3_9ROSA,1.046000e-19,901,38.9,0.389,85,2,205,207,...,NaN,NaN,AFDB,UniProt,0.057971,"[('K', 66, 'A'), ('K', 77, 'G'), ('K', 96, 'L'...",90.500000,Thaumatin-I_1LR2___W9SSV3,ATFTVINKCSHTVWPGILSNADSARLDSTGFELPPGGSRSFQSPPN...,Thaumatin-I_1LR2___W9SSV3


# Analyse monomer predictions – predicted using cropped MSAs and input sequences of sweet protein hits

In [ ]:
# Make output directory for TM-scores if it doesn't exist
tm_score_output_dir = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/tm_scores/"
if not os.path.exists(tm_score_output_dir):
    os.makedirs(tm_score_output_dir)

# Calculate TM-scores for all representative structures and save to CSV
tm_score_path = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/monomer_tm_scores.csv"
if not os.path.exists(tm_score_path):
    for index, row in tqdm(representative_structures_df.iterrows(), total=len(representative_structures_df)):
        target = row["pdb"].split("___")[1]
        query = row["pdb"].split("___")[0]
        target = target.replace("+", "_")
        pdb = row["pdb"].replace("+", "_")
        json_folder = f"{monomer_path}/{row['query']}/{pdb}"
        json_paths = [os.path.join(json_folder, f) for f in os.listdir(json_folder) if f.endswith("_confidences_aggregated.json")]
        best_model = get_best_of3_model(json_paths)
        pdb1 = f"{repo_path}/1_viridiplantae_prot_db/4_sweet_proteins/cleaned_pdbs/{row['query']}.pdb"
        pdb2 = f"{monomer_path}/{query}/{pdb}/{best_model}"
        output_file = f"{tm_score_output_dir}/{pdb}_tm_score.txt"
        if os.path.exists(pdb2):
            tm_score, rmsd, start, end = calculate_tm_score(pdb1, pdb2, output_file, norm_length=1)
            representative_structures_df.at[index, "of3_tm_score"] = tm_score
            representative_structures_df.at[index, "of3_RMSD"] = rmsd
            representative_structures_df.at[index, "of3_tm_align_start"] = start
            representative_structures_df.at[index, "of3_tm_align_end"] = end
            representative_structures_df.at[index, "of3_avg_pLDDT"] = avg_plddt(pdb2, mode = "CA")
            representative_structures_df.at[index, "best_model"] = best_model
            with open(output_file, "r") as f:
                output = f.readlines()
                for line in output:
                    if line.startswith("Aligned length="):
                        aligned_length = line.split(",")[0].split("= ")[1].strip()
                        align_seq_id = line.split(",")[2].split("= ")[1].strip()
                        representative_structures_df.at[index, "of3_tm_align_length"] = aligned_length
                        representative_structures_df.at[index, "of3_tm_align_seq_id"] = align_seq_id

        else:
            print(f"Warning: PDB file {pdb2} does not exist. Skipping TM-score calculation for {row['query']} and {target}.")
    cols = ['query', 'target', 'of3_tm_score', 'of3_RMSD','of3_tm_align_start', 'of3_tm_align_end',
       'of3_avg_pLDDT', 'of3_tm_align_seq_id', 'of3_tm_align_length', 'model', 'source', 'length_diff', 'interface_residues',
       'grantham_score', 'pdb', 'cropped_tseq', 'best_model']
    tm_scores_df = representative_structures_df[cols].copy()
    tm_scores_df.to_csv(tm_score_path, index=False)
else:
    tm_scores_df = pd.read_csv(tm_score_path)

# Keep only those with of3_avg_pLDDT > 70 and of3_tm_score > 0.7
print("Number of representative structures before filtering:", len(tm_scores_df))
tm_scores_df = tm_scores_df[tm_scores_df["of3_avg_pLDDT"] > 70]
tm_scores_df = tm_scores_df[tm_scores_df["of3_tm_score"] >= 0.7]
print("Number of representative structures after filtering:", len(tm_scores_df))

100%|██████████| 3676/3676 [04:06<00:00, 14.89it/s]

Number of representative structures before filtering: 3676
Number of representative structures after filtering: 3630


# Analyse complex predictions – prioritise complex predictions based on prior evidence on sweet protein binding sites

In [87]:
# Interaction site dict for the sweet protein-receptor heterodimer complexes
# These sites will be used to filter out predictions where the sweet protein is not predicted to interact with the receptor at all, and to focus analyses on the relevant interaction sites
interaction_site_dict = {
    "Honey_truffle_sweetener": {"ts1r2_human": np.arange(470, 541), "ts1r3_human": np.arange(470, 541)},
    "MNEI_1IV7": {"ts1r2_human": np.arange(1,541), "ts1r3_human": np.arange(0,0)},
    "Brazzein_4HE7": {"ts1r2_human": np.arange(0,0), "ts1r3_human": np.arange(470, 541)},
    "Miraculin": {"ts1r2_human": np.arange(428,475), "ts1r3_human": np.arange(0,0)},
    "Thaumatin-I_1LR2": {"ts1r2_human": np.arange(0,0), "ts1r3_human": np.arange(470, 541)},
    "Thaumatin-II_3AOK": {"ts1r2_human": np.arange(0,0), "ts1r3_human": np.arange(470, 541)},
}
# NOTE: ts1r2_human is chain A and ts1r3_human is chain B in the complex predictions
# The signal peptides have not been modelled. This has been accounted in the residue numbering of the interaction sites.

## Check whether predicted complexes interact with the specified residues ranges 
This is done to filter out proteins that are never docked near the suggested binding site.

In [88]:
contacts_csv = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/complex_contacts.csv"
complex_path = Path(f"{repo_path}/6_structural_analyses/4_openfold3_predictions/complexes")
if not os.path.exists(contacts_csv):
    def process_single_pdb(pdb_path_str):
        pdb_path = Path(pdb_path_str)    
        sweet_protein = pdb_path.parent.parent.name
        pred_dir = pdb_path.parent.name
        pdb_file = pdb_path.name
        contacts_AC = get_contacts(str(pdb_path), "A", "C")
        contacts_BC = get_contacts(str(pdb_path), "B", "C")
        return {
            "sweet_protein": sweet_protein,
            "prediction": pred_dir,
            "pdb_file": pdb_file,
            "tas1r2_contacts": contacts_AC,
            "tas1r3_contacts": contacts_BC
        }

    all_pdb_paths = [str(p) for p in complex_path.glob("*/*/*.pdb")]
    contacts = Parallel(n_jobs=-1, backend="loky")(
        delayed(process_single_pdb)(pdb) for pdb in tqdm(all_pdb_paths, desc="Processing PDBs")
    )
    contacts_df = pd.DataFrame(contacts)
    contacts_df.to_csv(contacts_csv, index=False)
else:
    contacts_df = pd.read_csv(contacts_csv)

# Loop through the contacts and filter out predictions where there are no contacts between the sweet protein and the receptor, and focus on the interaction sites defined in the interaction_site_dict
contacts_df["interaction"] = False
contacts_df["tas1r2_contacts_filtered"] = None
contacts_df["tas1r3_contacts_filtered"] = None
for index, row in contacts_df.iterrows():
    sweet_protein = row["sweet_protein"]
    tas1r2_contacts = [i[0] for i in ast.literal_eval(row["tas1r2_contacts"])]
    tas1r3_contacts = [i[0] for i in ast.literal_eval(row["tas1r3_contacts"])]

    # Filter contacts to only include those within the defined interaction sites
    tas1r2_interaction_sites = interaction_site_dict[sweet_protein]["ts1r2_human"]
    tas1r3_interaction_sites = interaction_site_dict[sweet_protein]["ts1r3_human"]
    tas1r2_contacts_filtered = [contact for contact in tas1r2_contacts if contact in tas1r2_interaction_sites]
    tas1r3_contacts_filtered = [contact for contact in tas1r3_contacts if contact in tas1r3_interaction_sites]
    if len(tas1r2_contacts_filtered) == 0 and len(tas1r3_contacts_filtered) == 0:
        contacts_df.at[index, "interaction"] = False
    else:
        contacts_df.at[index, "interaction"] = True

    contacts_df.at[index, "tas1r2_contacts_filtered"] = tas1r2_contacts_filtered
    contacts_df.at[index, "tas1r3_contacts_filtered"] = tas1r3_contacts_filtered

# For each "prediction", count how many interaction == True predictions there are
interaction_counts = contacts_df.groupby("prediction")["interaction"].sum().reset_index()
interaction_counts["protein"] = interaction_counts["prediction"].apply(lambda x: x.split("___")[1])
interaction_counts["target"] = interaction_counts["prediction"].apply(lambda x: x.split("___")[0])

# For each sweet protein, check how many times Native predictions have been docked in the right region
fails = []
for index, row in interaction_counts[interaction_counts["protein"] == "Native"].iterrows():
    print(f"Sweet protein: {row['target']}, native interactions: {row['interaction']}")
    for index2, row2 in interaction_counts[interaction_counts["target"] == row['target']].iterrows():
        if row2["interaction"] < row["interaction"]:
            fails.append(row2["prediction"])

print(f"Found {len(fails)} predictions that have fewer interactions than the native prediction: {fails}")

Sweet protein: Brazzein_4HE7, native interactions: 3
Sweet protein: Honey_truffle_sweetener, native interactions: 2
Sweet protein: MNEI_1IV7, native interactions: 5
Sweet protein: Miraculin, native interactions: 1
Sweet protein: Thaumatin-II_3AOK, native interactions: 4
Sweet protein: Thaumatin-I_1LR2, native interactions: 3
Found 1229 predictions that have fewer interactions than the native prediction: ['Brazzein_4HE7___A0A078GY26', 'Brazzein_4HE7___A0A087GR70', 'Brazzein_4HE7___A0ABM0YUX4', 'Brazzein_4HE7___A0ABM1RQB5', 'Brazzein_4HE7___A0ABQ7DJF1', 'Brazzein_4HE7___A0ABQ7ZQ79', 'Brazzein_4HE7___M4F9E7', 'Brazzein_4HE7___OneKP_scaffold-IPWB-2008469-Brassica_nigra', 'Honey_truffle_sweetener___A0A164ZZU0', 'Honey_truffle_sweetener___A0A1D6MXJ8', 'Honey_truffle_sweetener___A0A1R3K8F3', 'Honey_truffle_sweetener___A0A5A7P1H1', 'Honey_truffle_sweetener___A0A5B6U6J4', 'Honey_truffle_sweetener___A0A5D2GN07', 'Honey_truffle_sweetener___A0A5D2V2W4', 'Honey_truffle_sweetener___A0A5D2ZE14', 'Hon

# Choose predictions for SurfDiff analyses

In [91]:
surfdiff_structure_dir = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/surfdiff_structures"
source_dir = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/monomers"
if not os.path.exists(surfdiff_structure_dir):
    os.makedirs(surfdiff_structure_dir, exist_ok=True)

# Drop pdbs that are in fails list
surfdiff_df = tm_scores_df[~tm_scores_df["pdb"].isin(fails)].copy()
surfdiff_df.to_csv(f"{repo_path}/6_structural_analyses/4_openfold3_predictions/surfdiff_input_structures.csv", index=False)
surfdiff_df["query"].value_counts()

query
Thaumatin-I_1LR2           810
Honey_truffle_sweetener    608
Miraculin                  588
MNEI_1IV7                  232
Thaumatin-II_3AOK          170
Brazzein_4HE7               18
Name: count, dtype: int64

In [ ]:
# Replace "+" with "_" in the "pdb" column to match the file names in the monomers folder
surfdiff_df["pdb"] = surfdiff_df["pdb"].apply(lambda x: x.replace("+", "_"))

# Now, save the structures for SurfDiff analyses under "pdb_dir" folder
for _, row in surfdiff_df.iterrows():
    interaction = row["pdb"]
    pdb_id = row["pdb"].split("___")[1]
    if "Native" in interaction:
        continue
    best_model = row["best_model"]
    sweet_protein = row["query"]
    grantham_score = row["grantham_score"]
    src_path = f"{source_dir}/{sweet_protein}/{interaction}/{best_model}"
    os.makedirs(f"{surfdiff_structure_dir}/{sweet_protein}/pdb_dir", exist_ok=True)
    dest_path = f"{surfdiff_structure_dir}/{sweet_protein}/pdb_dir/{pdb_id}.pdb"
    
    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)
    else:
        print(f"Warning: Source PDB file {src_path} does not exist. Skipping copy for {interaction}.")

# Copy native structures to the same folder
for sweet_protein in surfdiff_df["query"].unique():
    native_pdb = f"{repo_path}/1_viridiplantae_prot_db/4_sweet_proteins/cleaned_pdbs/{sweet_protein}.pdb"
    dest_path = f"{surfdiff_structure_dir}/{sweet_protein}/pdb_dir/{sweet_protein}.pdb"
    if os.path.exists(native_pdb):
        shutil.copy(native_pdb, dest_path)

# Make SurfDiff input JSON
jsons = {}
for sweet_protein in surfdiff_df["query"].unique():
    jsons[sweet_protein] = {}
    jsons[sweet_protein]["query-id"] = sweet_protein
    pdb_files = os.listdir(f"{surfdiff_structure_dir}/{sweet_protein}/pdb_dir")
    pdb_files = [f.replace(".pdb", "") for f in pdb_files if f.endswith(".pdb")]
    if sweet_protein in pdb_files:
        pdb_files.remove(sweet_protein)
    jsons[sweet_protein]["subject-ids"] = pdb_files

# Save each json 
for sweet_protein, data in jsons.items():
    json_path = f"{surfdiff_structure_dir}/{sweet_protein}_input.json"
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)